### ***RAG Application***
#### **End-to-End Rag Application**

In [200]:
### load the Environment Variables

from dotenv import load_dotenv
load_dotenv(override=True)

True

In [201]:
### Check/Validate the path to load the files from the Document
import os

path = "../kubernetes"

if os.path.exists(path):
    print("Path is valid")
else:
    print("Path is not valid")

Path is valid


In [202]:
#load the documents using DirectoryLoader 

from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()

print("Number Of Documents:",len(documents))

Number Of Documents: 3983


In [203]:
### create a chunks using splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [204]:
###BM25 Retriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k=10

In [205]:
####Initalize the embedding model

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model = "BAAI/bge-large-en-v1.5",
    model_kwargs = {"device":"cpu"},
    encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [206]:
### load the vcctors from Chroma DB
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)


In [207]:
#### retriever for similarity search.
vector_retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":10}
)

In [208]:
### Hybrid Search
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights = [0.7,0.3]
)

In [209]:
### Lets test with hybrid retriever and how many candidates are retrieved
docs = hybrid_retriever.invoke("What is Kubernetes Deployment?")
print(len(docs))

20


In [210]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [211]:
### Get top 5 final document after reranking

def retrieve_and_rerank(query, k=5):

    retrieved_docs = hybrid_retriever.invoke(query)


    pairs = [[query,doc.page_content] for doc in retrieved_docs]

    scores = reranker.predict(pairs)

    ##zip the scores and retrieved order the documents based on score from highest to lowest

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    ## Get top k documents using ranked_docs and k value
    top_docs = [doc for (doc,score) in ranked_docs[:k]]

    return top_docs

In [212]:
### Format without unnecessary context

def build_context(documents):

    context = ""

    for i,doc in enumerate(documents,start=1):
        source = doc.metadata.get("source")
        source = source.replace("\\","/")
        source = source.split("/")[-1]
        #print(source)
        context+=f"""
        
    Source {source}
    Page: {doc.metadata.get("page")}
    Content: {doc.page_content}
        """
    return context



In [213]:
### Design a prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the context does not contain the answer, say:
   "I don't have information based on the provided documents."
4. Keep the answer concise and directly relevant.
5. List the sources and page numbers used at the very end of your response.
6. Format the sources strictly like this:
Sources:
- source: [Source Name/File] - page: [Page Number]

Context:
{context}

Question:
{question}

Answer:
""")

In [214]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021E3671B950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021E36899A00>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [215]:
from langchain_core.tracers import LangChainTracer

custom_tracer = LangChainTracer(project_name="RAG Application")

In [216]:
from langchain_core.tracers import context
def rag(query):
    config = {"run_name":"Rag Application","callbacks": [custom_tracer]}
    top_docs = retrieve_and_rerank(query)
    context = build_context(top_docs)
    message = prompt.invoke({"question":query,"context":context},config=config)
    answer = llm.invoke(message).content
    return answer
    

In [217]:
answer = rag("What is a Kubernetes Deployment?")
print(answer)


A Kubernetes Deployment is an object that defines the desired state for an application—specifying how many replicas of a Pod should run, how they should be created and updated, and how they should be scaled. The Deployment controller continuously monitors the Pods, schedules them onto Nodes, and replaces any failed instances, providing a self‑healing mechanism for the application.

Sources:
- source: Tutorials.pdf - page: 9
- source: Concepts.pdf - page: 5
- source: Tutorials.pdf - page: 2


In [218]:
test_queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "What is a ConfigMap?"
]

for query in test_queries:
    print("Query:",query)
    answer = rag(query)
    print("Response:",answer)
    print("*"*60)

Query: What is a Kubernetes Deployment?
Response: A Kubernetes Deployment is an object that tells the control plane how to create, update, scale, and manage instances of an application (Pods). It defines the desired state—such as the number of replicas—and the system continuously works to match that state, replacing failed pods and handling rollouts.  

Sources:
- source: Tutorials.pdf - page: 9
- source: Concepts.pdf - page: 5
- source: Tutorials.pdf - page: 2
************************************************************
Query: What is a Kubernetes Pod?
Response: A Kubernetes Pod is the smallest deployable unit in Kubernetes—a group of one or more containers that share storage, network resources, and a specification for how to run them, effectively acting as a logical host for the application containers.

Sources:
- source: Concepts.pdf - page: 85
- source: Tutorials.pdf - page: 12
************************************************************
Query: What is a Kubernetes Service?
Respons

In [219]:
test_queries = [
    "What happens when a Deployment specifies three replicas?",
    "What happens if one of the Pods managed by a Deployment fails?",
    "How does Kubernetes ensure that the actual state matches the desired state?",
    "How does a Service identify which Pods should receive traffic?",
    "How do containers within the same Pod communicate with each other?",
    "What is the difference between a Pod and a Deployment?",
    "What is the difference between a Deployment and a ReplicaSet?",
    "How are Deployments, ReplicaSets, and Pods related?",
    "How does a Deployment use ReplicaSets to manage Pods?",
    "How does a Service provide access to Pods managed by a Deployment?",
    "How can I run multiple copies of the same application?",
    "How can other applications find and communicate with my Pods?",
    "If I want three copies of my application running, which Kubernetes object should I use?",
    "If a Pod is deleted manually, how does Kubernetes respond when it is managed by a ReplicaSet?",
    "Which Kubernetes component continuously works to reconcile desired state and actual state?"
]

In [220]:
for query in test_queries:
    print("Query:",query)
    answer = rag(query)
    print("Response:",answer)
    print("*"*60)

Query: What happens when a Deployment specifies three replicas?
Response: When a Deployment’s spec sets `replicas: 3`, Kubernetes tries to create three Pods of that application and continuously works to keep three instances running, updating the Deployment’s status to reflect the desired state. (If a quota limits pods, fewer may be created.)  

Sources:
- source: Concepts.pdf - page: 5
- source: Tasks.pdf - page: 103
************************************************************
Query: What happens if one of the Pods managed by a Deployment fails?
Response: When a Pod that is part of a Deployment fails, the Deployment’s controller detects the failure and creates a replacement Pod (through its ReplicaSet). The new Pod is then scheduled onto a healthy node, ensuring the desired number of Pods is maintained.  

Sources:
- source: Concepts.pdf - page: 87
************************************************************
Query: How does Kubernetes ensure that the actual state matches the desired st